# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (see below).

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. This will let us access record sets and associated fields for further exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load metadata
dataset = mlc.Dataset(croissant_url)

# Print summary metadata
print("Dataset name: {}".format(dataset.metadata.name))
print("Description: {}".format(dataset.metadata.description))
print("Published:   {}".format(getattr(dataset.metadata, 'datePublished', None)))
print("License:     {}".format(getattr(dataset.metadata, 'license', None)))

## 2. Data Overview
List available record sets, fields, and their `@id`s. All further references use the `@id` as required for reproducibility.

We'll print the available record sets, then fields/columns for each record set, using their `@id` values.

In [ ]:
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s).\n")

for rs in record_sets:
    print(f"Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', None)}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - name: {field.name} (id: {field.id}, type: {getattr(field, 'data_type', None)})")
    print("")

## 3. Data Extraction
Load data from a selected record set into a pandas DataFrame for analysis. Below, we'll:
1. List all record set `@id`s.
2. Extract all available record sets and load them into DataFrames, keyed by their `@id`s.
3. Show the column names and a sample of the first record set.

You can adjust the selected record set and fields by referencing their `@id`s as printed above.

In [ ]:
# List of all record set `@id`s
record_set_ids = [rs.id for rs in dataset.record_sets]
print(f"Record set IDs found:\n{record_set_ids}\n")

dataframes = {}

for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Loaded {len(df)} records for record set {rsid}.")

# For demonstration, use the first record set
if record_set_ids:
    example_rs_id = record_set_ids[0]
    print(f"Columns in record set {example_rs_id}:")
    print(dataframes[example_rs_id].columns.tolist())
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's conduct some simple exploratory analysis:
- Select a *numeric* field (by its `@id`) and filter records
- Normalize the field
- If available, group by a categorical field and aggregate

Notes:
- Make sure to replace the field and group `@id`s below with appropriate values from the Data Overview section.

In [ ]:
# Select which record set and fields to analyze (replace with chosen @id)
record_set_id = example_rs_id  # Using first record set here
df = dataframes[record_set_id]

# Find a numeric field (by inspecting columns)
possible_numeric_fields = [col for col in df.columns if df[col].dtype in ('float64', 'int64')]
print(f"Numeric fields detected: {possible_numeric_fields}")
if not possible_numeric_fields:
    print("No numeric fields detected in this record set.")
else:
    # Pick the first numeric field
    numeric_field = possible_numeric_fields[0]

    # Set a threshold (e.g., above the median)
    threshold = df[numeric_field].median()

    # Filter and normalize
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field!r} > {threshold}: {len(filtered_df)} records")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Pick a string/categorical field to group by
    possible_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    print(f"Possible group fields: {possible_group_fields}")
    if possible_group_fields:
        group_field = possible_group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
        display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the numeric field and, if possible, the grouping field relationships.

Replace field ids in plots to tune for your dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not possible_numeric_fields:
    print('No numeric fields to visualize.')
else:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution: {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group if group field exists
    if possible_group_fields:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and explore the FAIR² dataset's record sets and fields using their Croissant `@id`s. We demonstrated loading data, filtering by numeric fields, normalizing and grouping data, and creating basic visualizations. For deeper exploration, reference the `@id` fields in your queries and see the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/) for further details.

You can extend this notebook with your own analyses, models, or export steps as required.